# Bodhan Tool Server & Autonomous Task Agent Walkthrough

This notebook demonstrates the **Bodhan MCP Server** and **Task Agent** built for the Bodhan AI Cookbook.

### Core Capabilities:
1. **MCP Tool Server** (`python -m bodhan_mcp`): Exposes Bodhan Indic models as standard Model Context Protocol tools over `stdio`.
2. **Autonomous Task Agent** (`python -m bodhan_agent`): Multi-step planner executing Indian language workflows with self-verification, budgeting, and honest failure on impossible requests.

In [ ]:
# Step 1: Inspect registered tools and MCP resource library
from bodhan_mcp.server import mcp_server, get_voice_library
import json

tool_names = list(mcp_server._tool_manager._tools.keys())
print("Registered MCP Tools:")
for name in tool_names:
    tool_info = mcp_server._tool_manager._tools[name]
    print(f"- {name}: {tool_info.description[:80]}...")

print("\nIndic-Speak Voice Library Resource Preview:")
library = json.loads(get_voice_library())
print(f"Supported Languages: {list(library['supported_languages'].keys())}")
print(f"Sample Voices: {library['voices'][:4]}")

In [ ]:
# Step 2: Test Tool Chunking & Limits Handling (Speech Synthesis -> Indic-Speak)
from bodhan_mcp.models import SpeakInput
from bodhan_mcp.tools.speak import run_speak
from bodhan_mcp.client import BodhanClient

client = BodhanClient()
speak_args = SpeakInput(
    text="வணக்கம்! போதன் ஏஐ மூலம் தமிழ் ஒலி சுருக்கம் உருவாக்கப்படுகிறது. இது பல வாக்கியங்களைக் கொண்ட உரை.",
    language="ta",
    output_audio_path="./examples/demo_outputs/walkthrough_sample.wav"
)
speak_res = run_speak(speak_args, client)
print(f"Generated Audio File: {speak_res.audio_path}")
print(f"Duration: {speak_res.duration_seconds}s | Characters: {speak_res.character_count} | Cost: ₹{speak_res.metadata.cost_inr:.4f} INR")

In [ ]:
# Step 3: Run the Task Agent on a Multi-step Educational Task
from bodhan_agent.agent import TaskAgent

task = "Make a two-minute Tamil audio summary of this chapter"
agent = TaskAgent(
    task=task,
    files_path="sample_inputs/chapter.pdf",
    out_dir="./examples/demo_outputs/walkthrough_run",
    max_steps=10,
    max_cost_inr=10.0
)
report = agent.run()
print(f"Agent Run Status: {report.status}")
print(f"Total Steps: {report.total_steps} | Cost: ₹{report.total_cost_inr:.4f} INR | Time: {report.total_duration_seconds}s")
print(f"Verifications Run: {[v.check_type + ': ' + ('Passed' if v.passed else 'Failed') for v in report.verifications]}")

In [ ]:
# Step 4: Run the Task Agent on an Impossible Task (Honest Failure demonstration)
impossible_task = "Translate this video into French"
agent_fail = TaskAgent(
    task=impossible_task,
    files_path="sample_inputs/video.mp4",
    out_dir="./examples/demo_outputs/walkthrough_fail",
)
fail_report = agent_fail.run()
print(f"Status: {fail_report.status}")
print(f"Explanation: {fail_report.explanation}")

In [ ]:
# Step 5: Read and Display the Generated result.md and trace.jsonl
with open("./examples/demo_outputs/walkthrough_run/result.md", "r", encoding="utf-8") as f:
    print(f.read())